# 04 · Simulated annealing

Hill climbing gets stuck on the **nearest** hill because it never takes a step down. **Simulated annealing (SA)** does the same thing, with one change: it sometimes keeps a **worse** move on purpose, so it can walk down off a small hill and find a bigger one.

The name comes from metalworking. *Annealing* means heating metal and cooling it slowly so its atoms settle into a strong structure. The algorithm copies that:

- **Hot** (at the start): worse moves are accepted often, so the search wanders and explores.
- **Cooling**: worse moves are accepted less and less.
- **Cold** (at the end): almost no worse moves are accepted, so it behaves exactly like hill climbing and polishes the best area it found.

**A common misunderstanding:** the temperature is *not* a changing `MAX_STALL`. It doesn't decide how many tries are allowed; it decides **how much worse a move may be and still be kept**. It's measured in score points.

All the code in this notebook is complete. Read it, run it, and change the settings.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research').is_dir())
sys.path.insert(0, str(ROOT))

import math
import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from research.evaluate import score, summary, validate
from research.local_search import Schedule
from research.reference.greedy import run_greedy
from research.reference.hill_climbing import MOVE_WEIGHTS, MOVES, hill_climb
from research.reference.simulated_annealing import estimate_start_temperature
from research.synthetic import Scenario, generate

data = generate(seed=1)
greedy_result = run_greedy(**data)
print('Greedy score:', score(greedy_result, data).total)

## 1. The accept rule

| | Better or equal move | Worse move |
|:---|:---|:---|
| Hill climbing | keep | always undo |
| Simulated annealing | keep | keep **with probability** `e^(Δ / T)` |

- **Δ** (delta) = new score − old score. For a worse move it is negative, e.g. −20.
- **T** = the current temperature.
- `e^(Δ / T)` is always between 0 and 1, so it works as a probability. To decide, draw a random number between 0 and 1 and keep the move if the number is smaller.

In [ ]:
def sa_accept(current, candidate, temperature, rng):
    """Keep a move that is at least as good; keep a worse one with probability e^(delta / T)."""
    if candidate >= current:
        return True                                  # better or equal: always keep
    if temperature <= 0:
        return False                                 # frozen: behave like hill climbing
    probability = math.exp((candidate - current) / temperature)
    return rng.random() < probability                # roll the dice

How likely is a worse move to be kept? Each cell is the chance for a move that loses Δ points at temperature T:

- **Small loss, high temperature**: kept often.
- **Big loss, low temperature**: almost never.

In [ ]:
losses = [5, 20, 50, 100, 200]
temperatures = [200, 100, 50, 10, 1]
pd.DataFrame(
    {f'T = {t}': [f'{math.exp(-loss / t):.0%}' for loss in losses] for t in temperatures},
    index=[f'loses {loss}' for loss in losses],
)

## 2. The starting temperature: measure it, don't guess it

Whether T = 100 is "hot" depends on how big score changes are in *this* problem. So we measure:

1. Try a few hundred random moves on the greedy schedule and note how many points the worse ones lose.
2. Take a typical (median) loss.
3. Pick T so that such a loss is accepted with a chosen probability, here 50%: `e^(-loss / T) = 0.5`, so `T = loss / ln(2)`.

`START_ACCEPT_PROBABILITY` is the knob: 0.8 starts hotter (more exploring), 0.2 starts cooler.

In [ ]:
START_ACCEPT_PROBABILITY = 0.5
END_TEMPERATURE = 1.0

t0 = estimate_start_temperature(data, greedy_result, accept_probability=START_ACCEPT_PROBABILITY)
print(f'Starting temperature: {t0:.1f}')
print(f'A move losing {t0 * math.log(2):.0f} points is kept {START_ACCEPT_PROBABILITY:.0%} of the time at the start.')
print(f'At the end (T = {END_TEMPERATURE}), losing 5 points is kept {math.exp(-5 / END_TEMPERATURE):.1%} of the time.')

## 3. Cooling: how the temperature drops

You choose a **start** temperature, an **end** temperature and a **budget** (a number of tries, or seconds). The cooling schedule fills in the path between them.

- **Linear**: drops by the same amount every second. From 100 to 5 over 5 seconds, that's −19 per second.
- **Geometric** (the usual choice): multiplies by the same factor every step, so it drops fast at first and slowly at the end. The algorithm spends more time at low temperatures, polishing.

Meanwhile the algorithm makes **thousands of tries per second**. Each try just checks the temperature at that moment.

In [ ]:
def temperature_at(fraction, t_start, t_end):
    """Geometric cooling over a budget: fraction = share of the budget used (0 at the start, 1 at the end)."""
    return t_start * (t_end / t_start) ** fraction


seconds = [s / 10 for s in range(51)]
plt.figure(figsize=(8, 3.5))
plt.plot(seconds, [100 - 19 * s for s in seconds], label='linear: -19 per second')
plt.plot(seconds, [temperature_at(s / 5, 100, 5) for s in seconds], label='geometric: x the same factor')
plt.xlabel('seconds')
plt.ylabel('temperature')
plt.title('From T = 100 to T = 5 in 5 seconds')
plt.legend()
plt.show()

## 4. The simulated annealing loop

The same loop as hill climbing in notebook 03, with three differences:

1. `sa_accept` instead of `accept`.
2. The temperature is recalculated from the time used.
3. It keeps a copy of the **best schedule ever seen** (`best_snapshot`), because the current schedule can get worse. At the end it returns the best one, not the last one.

It runs for a fixed time instead of stopping after `MAX_STALL` failures.

In [ ]:
TIME_LIMIT_S = 3.0


def my_simulated_annealing(data, start_result, time_limit_s=TIME_LIMIT_S,
                           start_accept_probability=START_ACCEPT_PROBABILITY,
                           end_temperature=END_TEMPERATURE, seed=0):
    rng = random.Random(seed)
    names = list(MOVES)
    weights = [MOVE_WEIGHTS[name] for name in names]

    schedule = Schedule(data, start_result)
    current = schedule.total()
    best, best_snapshot = current, schedule.snapshot()

    t_start = estimate_start_temperature(data, start_result, accept_probability=start_accept_probability, seed=seed)
    t_end = min(end_temperature, t_start)
    trace = {'current': [], 'best': [], 'temperature': [], 'worse': []}
    started = time.perf_counter()

    while True:
        fraction = (time.perf_counter() - started) / time_limit_s
        if fraction >= 1:
            break
        temperature = temperature_at(fraction, t_start, t_end)

        name = rng.choices(names, weights=weights)[0]
        change = MOVES[name](schedule, rng)
        worse = None                                   # for the chart: None = not a worse move
        if change is not None and schedule.apply(change):
            candidate = schedule.total()
            if candidate < current:
                worse = 0                              # a worse move... (0 = undone)
            if sa_accept(current, candidate, temperature, rng):
                if candidate < current:
                    worse = 1                          # ...that was kept
                current = candidate
                if current > best:
                    best, best_snapshot = current, schedule.snapshot()
            else:
                schedule.undo(change)

        trace['current'].append(current)
        trace['best'].append(best)
        trace['temperature'].append(temperature)
        trace['worse'].append(worse)

    schedule.restore(best_snapshot)
    return schedule, trace


started = time.perf_counter()
schedule, trace = my_simulated_annealing(data, greedy_result)
result = schedule.to_result('simulated_annealing', started)

hc_schedule, hc_history, _ = hill_climb(data, greedy_result)
print('Problems:', validate(result, data) or 'none')
print(f"Greedy {score(greedy_result, data).total}   hill climbing {hc_history[-1]}   simulated annealing {score(result, data).total}")
print(f"{len(trace['current'])} tries in {TIME_LIMIT_S} s")

## 5. Watching it work

Three charts over the same run, read top to bottom:

1. **Current vs best score.** The current score zigzags up *and down* early on: those are the worse moves being kept. The best score only rises. Later the zigzag calms down.
2. **Temperature**, falling from hot to cold.
3. **Share of worse moves kept**: high at the start, close to zero at the end.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(trace['current'], label='current', alpha=0.7)
axes[0].plot(trace['best'], label='best so far', linewidth=2)
axes[0].axhline(hc_history[-1], color='grey', linestyle='--', label='hill climbing finished here')
axes[0].set_ylabel('score')
axes[0].legend()
axes[1].plot(trace['temperature'], color='#dc2626')
axes[1].set_ylabel('temperature')
axes[1].set_yscale('log')
kept = pd.Series(trace['worse'], dtype='float').rolling(500, min_periods=20).mean()
axes[2].plot(kept, color='#7c3aed')
axes[2].set_ylabel('worse moves kept')
axes[2].set_ylim(0, 1)
axes[2].set_xlabel('try')
plt.tight_layout()
plt.show()

In [ ]:
hc_result = hc_schedule.to_result('hill_climbing', time.perf_counter())
pd.DataFrame([summary(greedy_result, data), summary(hc_result, data), summary(result, data)],
             index=['greedy', 'hill climbing', 'simulated annealing']).drop(columns='runtime_ms')

## 6. Five runs: does it get stuck less?

The same test as notebook 03: five random seeds on the same problem. Hill climbing usually ends on several different scores (different small hills). Simulated annealing should end **higher** and **closer together**. Each run gets 1.5 seconds.

In [ ]:
rows = []
for run_seed in range(5):
    hc_final = hill_climb(data, greedy_result, seed=run_seed)[1][-1]
    sa_final = my_simulated_annealing(data, greedy_result, time_limit_s=1.5, seed=run_seed)[1]['best'][-1]
    rows.append({'seed': run_seed, 'hill climbing': hc_final, 'simulated annealing': sa_final})
five = pd.DataFrame(rows)
five

In [ ]:
five[['hill climbing', 'simulated annealing']].agg(['mean', 'min', 'max', 'std']).round(1)

## 7. Twenty problems, same time for everyone

A fair comparison gives every method **the same amount of time**. Otherwise SA might only win because it ran longer. Hill climbing uses its time by **restarting** with new random seeds and keeping its best run.

This cell takes about a minute.

In [ ]:
BUDGET_S = 1.0


def hill_climb_with_restarts(data, start_result, time_limit_s):
    started, best_score, run_seed = time.perf_counter(), None, 0
    while time.perf_counter() - started < time_limit_s:
        final = hill_climb(data, start_result, seed=run_seed, max_stall=500)[1][-1]
        best_score = final if best_score is None else max(best_score, final)
        run_seed += 1
    return best_score, run_seed


rows = []
for seed in range(20):
    problem = generate(seed=seed)
    start = run_greedy(**problem)
    hc_score, restarts = hill_climb_with_restarts(problem, start, BUDGET_S)
    sa_schedule, sa_trace = my_simulated_annealing(problem, start, time_limit_s=BUDGET_S)
    rows.append({
        'seed': seed,
        'greedy': score(start, problem).total,
        'hill climbing (restarts)': hc_score,
        'restarts': restarts,
        'simulated annealing': sa_trace['best'][-1],
        'valid': not validate(sa_schedule.to_result('sa', 0.0), problem),
    })
comparison = pd.DataFrame(rows)
comparison['SA wins'] = comparison['simulated annealing'] > comparison['hill climbing (restarts)']
comparison

In [ ]:
print(f"SA beat hill climbing on {comparison['SA wins'].sum()} of {len(comparison)} problems")
comparison[['greedy', 'hill climbing (restarts)', 'simulated annealing']].describe().round(1)

## 8. What you can tune

| Setting | What it does | Too high | Too low |
|:---|:---|:---|:---|
| `START_ACCEPT_PROBABILITY` (start temperature) | How freely it explores at first | Wanders randomly and wastes the early time | Barely explores, ends like hill climbing |
| `END_TEMPERATURE` | How strict it is at the end | Never settles; keeps accepting worse moves while it should polish | Fine, just a little time spent frozen |
| Cooling schedule | The path from hot to cold | | |
| Budget (`TIME_LIMIT_S` or tries) | How long it searches | Diminishing returns | Cools too fast ("quenching") and gets stuck |
| Moves and their weights | Which schedules it can reach | | |

Change the values below and run the cell. Try `START_ACCEPT_PROBABILITY` = 0.95 (too hot), 0.05 (too cold), and `TIME_LIMIT_S` = 0.3 (too fast) vs 6.

In [ ]:
settings = {'start_accept_probability': 0.5, 'end_temperature': 1.0, 'time_limit_s': 3.0}

tuned_schedule, tuned_trace = my_simulated_annealing(data, greedy_result, **settings)
print(settings, '->', tuned_trace['best'][-1])
plt.figure(figsize=(9, 3))
plt.plot(tuned_trace['current'], alpha=0.7, label='current')
plt.plot(tuned_trace['best'], linewidth=2, label='best so far')
plt.legend()
plt.show()

## Summary

- SA = hill climbing + **sometimes keep a worse move**, with probability `e^(Δ / T)`.
- The temperature starts high (explore) and cools down (polish). You set a start, an end and a budget.
- Measure the start temperature from the problem's typical score changes; don't guess it.
- Always keep the **best schedule seen**, because the current one can get worse.
- Compare methods with **the same time budget**.
- Tune the settings like hyperparameters (grid search, random search, Optuna) on **tuning problems**, and check them on **different** problems.

The full notes are in `notes/personal_notes.md`.

In [ ]:
import optuna

TUNING_SEEDS = range(100, 105)    # tuning problems, never used for the final check
TRIAL_BUDGET_S = 0.4              # SA time per problem in each trial
N_TRIALS = 30                     # about 2 seconds per trial, so about a minute in total

tuning_problems = []
for tuning_seed in TUNING_SEEDS:
    problem = generate(seed=tuning_seed)
    start = run_greedy(**problem)
    tuning_problems.append((problem, start, score(start, problem).total))


def objective(trial):
    """One trial: pick settings, run SA on every tuning problem, return the average gain over greedy."""
    settings = {
        'start_accept_probability': trial.suggest_float('start_accept_probability', 0.05, 0.95),
        'end_temperature': trial.suggest_float('end_temperature', 0.1, 20, log=True),
    }
    gains = []
    for problem, start, greedy_score in tuning_problems:
        _schedule, run = my_simulated_annealing(problem, start, time_limit_s=TRIAL_BUDGET_S, seed=0, **settings)
        gains.append(run['best'][-1] - greedy_score)
    return sum(gains) / len(gains)


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=0))
# Trial 0 = the hand-picked settings, so we can see whether Optuna beats them.
study.enqueue_trial({'start_accept_probability': START_ACCEPT_PROBABILITY, 'end_temperature': END_TEMPERATURE})
study.optimize(objective, n_trials=N_TRIALS)

print('Hand-picked settings:', study.trials[0].params, '-> average gain', round(study.trials[0].value, 1))
print('Best settings found: ', {k: round(v, 3) for k, v in study.best_params.items()},
      '-> average gain', round(study.best_value, 1))

In [ ]:
trials = study.trials_dataframe(attrs=('number', 'value', 'params')).rename(columns={
    'value': 'average gain',
    'params_start_accept_probability': 'start_accept_probability',
    'params_end_temperature': 'end_temperature',
})

plt.figure(figsize=(7, 4.5))
points = plt.scatter(trials['start_accept_probability'], trials['end_temperature'],
                     c=trials['average gain'], cmap='viridis', s=60)
plt.colorbar(points, label='average gain over greedy')
plt.yscale('log')
plt.xlabel('start_accept_probability')
plt.ylabel('end_temperature')
plt.title('Every Optuna trial: where it looked, and how well it did')
plt.show()

trials.sort_values('average gain', ascending=False).head(10).round(3)

### Does it hold up on new problems?

The settings were tuned on seeds 100–104. Now compare the hand-picked settings and Optuna's on **different** problems (seeds 0–9), with the same time budget. This is the "test set" check.

In [ ]:
tuned = study.best_params
rows = []
for seed in range(10):
    problem = generate(seed=seed)
    start = run_greedy(**problem)
    greedy_score = score(start, problem).total
    hand = my_simulated_annealing(problem, start, time_limit_s=TRIAL_BUDGET_S)[1]['best'][-1]
    auto = my_simulated_annealing(problem, start, time_limit_s=TRIAL_BUDGET_S, **tuned)[1]['best'][-1]
    rows.append({'seed': seed, 'hand-picked gain': round(hand - greedy_score, 1), 'Optuna gain': round(auto - greedy_score, 1)})
test = pd.DataFrame(rows)
print(f"Average gain over greedy on new problems: hand-picked {test['hand-picked gain'].mean():.1f}, "
      f"Optuna {test['Optuna gain'].mean():.1f}")
test

### What to take away from tuning

- **Tune on some problems, test on others.** The tuning used seeds 100–104; the check above used 0–9. If the settings only win on the tuning problems, they are overfitted, exactly like a model that only does well on its training data.
- **Settings belong to a budget.** Settings tuned for 0.4 seconds may not be best for 5 seconds. If the time limit changes, tune again.
- **Results are noisy.** SA is random, so a small difference between two settings can be luck. Averaging over several problems (and, with more time, several seeds per problem) makes the comparison trustworthy.
- **Tuning makes it better, not faster.** It finds better settings for the same time. Making each try faster (delta evaluation, faster code) is a separate, engineering job.

## Summary

- SA = hill climbing + **sometimes keep a worse move**, with probability `e^(Δ / T)`.
- The temperature starts high (explore) and cools down (polish). You set a start, an end and a budget.
- Measure the start temperature from the problem's typical score changes; don't guess it.
- Always keep the **best schedule seen**, because the current one can get worse.
- Compare methods with **the same time budget**.

The full notes are in `notes/personal_notes.md`.